# Module 14 - Preference tuning (DPO)

Use this notebook after `tests/test_dpo.py` is passing and after you have saved a Module 13 `*-SFT` model artifact. The notebook loads the strongest available SFT artifact by default, builds a small preference dataset, checks the step-0 `log(2)` invariant, trains with DPO, compares SFT vs DPO behavior, and saves a DPO artifact for Module 15.

DPO should feel like SFT with one extra dimension: every prompt has a chosen answer and a rejected answer, and the frozen reference model anchors how far the policy is allowed to move.

## Setup

In [ ]:
from pathlib import Path
import json
import math
import subprocess
import sys

import matplotlib.pyplot as plt
import torch

from g2c.artifacts import (
    available_model_artifacts_with_suffix,
    load_model_artifact_with_tokenizer,
    save_huggingface_model_artifact,
    save_model_artifact,
)
from g2c.dpo import (
    DPOTrainer,
    PreferenceExample,
    dpo_loss,
    pad_and_collate_pref,
    sequence_logprob,
)
from g2c.notebook_extras.dpo import plot_dpo_history, train_dpo_with_progress
from g2c.notebook_extras.model_selection import select_sft_artifact_name
from g2c.sampling import generate
from g2c.sft import ChatTemplate

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
print(repo_root)

Run the DPO tests before proceeding. In the clean scaffold this cell should fail until you implement the Module 14 TODOs in `g2c/dpo/`.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_dpo.py", "-q"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 14 DPO tests are not passing yet."

## Model selection

BaseLM-SFT is the default because DPO needs a model that already has useful instruction-following behavior. To run DPO on your own model, set `MODEL_SELECTION = "course"` for the strongest course-trained `*-SFT` artifact, or set it to a concrete artifact such as `"StoryLM-30M-SFT"`.

In [ ]:
MODEL_SELECTION = "BaseLM"  # "BaseLM", "course", or an SFT artifact name such as "TinyLLM-30M-SFT"
TRAIN_DEVICE = "auto"
SEED = 14

SFT_ARTIFACT_NAME = select_sft_artifact_name(MODEL_SELECTION, repo_root=repo_root)
print("selected SFT artifact:", SFT_ARTIFACT_NAME)

## Load the selected SFT artifact

DPO holds two model copies at once: a trainable policy and a frozen reference. If memory is tight, choose a smaller artifact or set `TRAIN_DEVICE = "cpu"` for the first debugging pass.

In [ ]:
available_sft = available_model_artifacts_with_suffix("-SFT", repo_root=repo_root)
if available_sft:
    print("available SFT artifacts:")
    for artifact in available_sft:
        print(f"  rank {artifact.rank:>3}: {artifact.name}")
else:
    print("No SFT artifacts found under artifacts/models/.")

policy_artifact = load_model_artifact_with_tokenizer(
    SFT_ARTIFACT_NAME,
    repo_root=repo_root,
    device=TRAIN_DEVICE,
)
reference_artifact = load_model_artifact_with_tokenizer(
    SFT_ARTIFACT_NAME,
    repo_root=repo_root,
    device=TRAIN_DEVICE,
)

policy_model = policy_artifact.model
ref_model = reference_artifact.model
tokenizer = policy_artifact.tokenizer
template = ChatTemplate()
pad_id = tokenizer.special_to_id.get("<|pad|>", getattr(tokenizer, "pad_token_id", None) or 0)
end_id = tokenizer.special_to_id.get(template.END, getattr(tokenizer, "eos_token_id", None))
tokenizer_vocab_size = len(getattr(tokenizer, "vocab", getattr(tokenizer, "inner", tokenizer)))

def model_device(model) -> torch.device:
    device = getattr(model, "device", None)
    if isinstance(device, torch.device):
        return device
    for parameter in model.parameters():
        return parameter.device
    return torch.device("cpu")


print("loaded SFT artifact:", policy_artifact.name)
print("display:", policy_artifact.display_name)
print("kind:", policy_artifact.manifest.get("kind", "course_transformer"))
print("model vocab:", policy_model.vocab_size)
print("tokenizer vocab:", tokenizer_vocab_size)
print("max seq len:", policy_model.max_seq_len)
print("pad id:", pad_id, "end id:", end_id)
print("policy device:", model_device(policy_model))

## Sampling helpers

These are notebook helpers, not the Module 14 deliverable. They render the same chat template used in SFT and stop generation at `<|end|>` when that token exists.

In [ ]:
def chat_prompt(user_text: str, assistant_prefix: str = "") -> str:
    return (
        template.render([{"role": "user", "content": user_text}])
        + f"{template.ASSISTANT}\n"
        + assistant_prefix
    )


def encode_for_model(model, text: str) -> torch.Tensor:
    ids = tokenizer.encode_with_vocab_size(text, model.vocab_size)
    if not ids:
        raise ValueError("prompt encoded to no tokens")
    return torch.tensor(ids, dtype=torch.long)


def sample_from_model(
    model,
    prompt: str,
    *,
    max_new_tokens: int = 80,
    temperature: float = 0.7,
    top_p: float | None = 0.9,
    seed: int = SEED,
) -> str:
    prompt_ids = encode_for_model(model, prompt)
    ids = generate(
        model,
        prompt_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=1.1,
        eos_id=end_id,
        generator=torch.Generator().manual_seed(seed),
    )
    return tokenizer.decode([int(x) for x in ids.tolist()])


def sample_response(model, user_text: str, **kwargs) -> str:
    return sample_from_model(model, chat_prompt(user_text), **kwargs)


def printable(text: str) -> str:
    has_control = any(ord(ch) < 32 and ch not in "\n\t" for ch in text)
    return text.encode("unicode_escape").decode("ascii") if has_control else text


def show_response(label: str, model, user_text: str, **kwargs) -> str:
    text = sample_response(model, user_text, **kwargs)
    print("-" * 72)
    print(label)
    print(printable(text))
    return text

A quick baseline read on the SFT model. This is not graded; it gives you a qualitative anchor before DPO changes the policy.

In [ ]:
BASELINE_PROMPTS = [
    "What is the capital of Russia?",
    "Context: SFT masks user tokens and trains on assistant tokens. Which tokens does SFT train on?",
    "Call calculator for 23 * 17.",
]

for prompt in BASELINE_PROMPTS:
    print("=" * 72)
    print("user:", prompt)
    show_response("SFT model", ref_model, prompt, max_new_tokens=80, seed=SEED)

## Exercise 1 - Build preference pairs

The starter set below is large enough to make validation less jumpy while still staying small enough for local experiments. It covers preference axes a small SFT model can plausibly move on: factual correctness, arithmetic, format following, honesty, grounded context use, debugging advice, concise answer style, tool-call JSON, and answering the actual question.

Course-specific ideas appear mostly inside grounded prompts where the context supplies the fact. That keeps the preference task focused on behavior instead of asking a small model to acquire a lot of new course knowledge from DPO alone.

The `sft_failure` rows are the most important rows pedagogically: their rejected answers look like common bad samples from the SFT model. DPO is most visible when the rejected completion is something the current policy actually likes too much.

Each row should vary one axis at a time. Avoid pairing a long polished chosen answer with a short broken rejected answer unless the preference you want to teach is length.

In [ ]:
def pref(kind: str, user: str, chosen: str, rejected: str) -> dict[str, str]:
    return {"kind": kind, "user": user, "chosen": chosen, "rejected": rejected}


preference_rows = [
    # factual correctness
    pref("factual", "What is the capital of France?", "Paris.", "London."),
    pref("factual", "What is the capital of Spain?", "Madrid.", "Lisbon."),
    pref("factual", "What is the capital of Italy?", "Rome.", "Milan."),
    pref("factual", "What is the capital of Germany?", "Berlin.", "Munich."),
    pref("factual", "What is the capital of Japan?", "Tokyo.", "Kyoto."),
    pref("factual", "Which planet is known as the red planet?", "Mars.", "Venus."),
    pref("factual", "What gas do plants take in for photosynthesis?", "Carbon dioxide.", "Oxygen."),
    pref("factual", "What do bees make?", "Honey.", "Milk."),
    pref("factual", "What planet is largest in our solar system?", "Jupiter.", "Mars."),
    pref("factual", "What does the Moon orbit?", "Earth.", "The Sun."),
    pref("factual", "What continent is Kenya in?", "Africa.", "Europe."),
    pref("factual", "What organ pumps blood?", "The heart.", "The lungs."),
    pref("factual", "What process lets plants make sugar?", "Photosynthesis.", "Respiration."),
    pref("factual", "Who wrote Hamlet?", "William Shakespeare.", "Charles Dickens."),
    pref("factual", "How many days are in a week?", "Seven.", "Ten."),
    pref("factual", "What do humans breathe in to survive?", "Oxygen.", "Carbon dioxide."),
    pref("factual", "What is the freezing point of water in Celsius?", "0 degrees Celsius.", "100 degrees Celsius."),
    pref("factual", "What is the largest ocean on Earth?", "The Pacific Ocean.", "The Atlantic Ocean."),
    # arithmetic
    pref("arithmetic", "What is 2 + 3?", "5.", "6."),
    pref("arithmetic", "What is 4 * 5?", "20.", "9."),
    pref("arithmetic", "What is 12 - 7?", "5.", "19."),
    pref("arithmetic", "What is 8 + 6?", "14.", "13."),
    pref("arithmetic", "What is 9 / 3?", "3.", "6."),
    pref("arithmetic", "What is 6 + 7?", "13.", "12."),
    pref("arithmetic", "What is 15 - 6?", "9.", "21."),
    pref("arithmetic", "What is 3 * 8?", "24.", "11."),
    pref("arithmetic", "What is 11 + 4?", "15.", "14."),
    pref("arithmetic", "What is 18 / 2?", "9.", "6."),
    pref("arithmetic", "What is 7 + 5?", "12.", "13."),
    pref("arithmetic", "What is 6 * 6?", "36.", "12."),
    pref("arithmetic", "What is 20 - 8?", "12.", "28."),
    pref("arithmetic", "What is 5 * 9?", "45.", "14."),
    # hard negatives from common SFT failure modes
    pref("sft_failure", "Which planet is known as the red planet?", "Mars.", "Africa."),
    pref("sft_failure", "What is 7 + 5?", "12.", "20."),
    pref("sft_failure", "Explain logits in one sentence.", "Logits are unnormalized scores before softmax turns them into probabilities.", "logits."),
    pref("sft_failure", "Explain RAG in one sentence.", "RAG retrieves relevant context and includes it in the prompt before generation.", "Rag."),
    pref("sft_failure", "Explain a tool call in one sentence.", "A tool call is structured text the runtime parses and executes.", "Tools like the word processor or the calculator that you need."),
    pref("sft_failure", "Explain a tool call in one sentence.", "A tool call is structured text the runtime parses and executes.", "inonesentence"),
    pref("sft_failure", "Context: Softmax turns logits into probabilities that sum to one. What does softmax produce?", "Probabilities that sum to one.", "Softmax."),
    pref("sft_failure", "Context: Softmax turns logits into probabilities that sum to one. What does softmax produce?", "Probabilities that sum to one.", "one."),
    pref("sft_failure", "If you are unsure about an answer, what should you say?", "Say what information is missing.", "I cannot give it."),
    pref("sft_failure", "What does SFT stand for?", "Supervised fine-tuning.", "Softmax turns logits into probabilities that sum to one."),
    pref("sft_failure", "What gas do plants take in for photosynthesis?", "Carbon dioxide.", "Africa."),
    pref("sft_failure", "What is the capital of Italy?", "Rome.", "Italy."),
    # format following
    pref("format", "Return exactly one color.", "Blue.", "Blue is a color that many people like."),
    pref("format", "Return exactly one number: seven.", "7.", "The number is 7, which is seven."),
    pref("format", "Answer yes or no: is water wet?", "Yes.", "Water can be considered wet in many contexts."),
    pref("format", "Answer with one word: cat or dog?", "Cat.", "I would choose cat because cats are nice."),
    pref("format", "Return lowercase only: HELLO.", "hello", "Hello is now lowercase: hello."),
    pref("format", "Return uppercase only: quiet.", "QUIET", "The uppercase version is QUIET."),
    pref("format", "Return JSON with color blue.", "{\"color\":\"blue\"}", "The JSON is {\"color\":\"blue\"}."),
    pref("format", "Return exactly two bullet items: red and blue.", "- red\n- blue", "Here are two colors:\n- red\n- blue\nThey are common colors."),
    pref("format", "Answer with one word: opposite of hot.", "Cold.", "The opposite of hot is cold."),
    pref("format", "Return only the filename: report.md", "report.md", "The filename is report.md."),
    pref("format", "Name one primary color and nothing else.", "Red.", "Primary colors are important in art."),
    pref("format", "Should the answer include extra explanation? Answer yes or no.", "No.", "Extra explanation can sometimes help."),
    pref("format", "Return only this word: tensor", "tensor", "The word is tensor."),
    pref("format", "Return CSV with two values: alpha and beta.", "alpha,beta", "alpha and beta"),
    pref("format", "Return JSON with animal cat.", "{\"animal\":\"cat\"}", "animal: cat"),
    pref("format", "Answer in one word: opposite of early.", "Late.", "The opposite of early is late."),
    pref("format", "Return exactly three numbers: 1, 2, 3.", "1, 2, 3", "The numbers are 1, 2, and 3."),
    pref("format", "Answer true or false: 5 is greater than 2.", "True.", "Yes, five is greater than two."),
    # honesty and uncertainty
    pref("honesty", "If you are not sure about an answer, what should you say?", "I am not sure.", "I know for certain."),
    pref("honesty", "If a fact might be outdated, what should an assistant do?", "Say it may need verification.", "State it confidently anyway."),
    pref("honesty", "Should an assistant invent a source when it lacks one?", "No, it should say it does not have a source.", "Yes, it should make one sound plausible."),
    pref("honesty", "If the prompt is ambiguous, what is a good response?", "Ask a clarifying question or state the assumption.", "Guess silently and pretend it was clear."),
    pref("honesty", "If you cannot see a file, what should you say?", "I cannot see that file unless you provide it.", "I checked the file and it looks correct."),
    pref("honesty", "If there is not enough information, what should you do?", "Say what information is missing.", "Fill in the missing details confidently."),
    pref("honesty", "Should an assistant claim a test passed if it did not run it?", "No, it should say the test was not run.", "Yes, it can say the test passed anyway."),
    pref("honesty", "If an answer depends on local files you have not read, what should you do?", "Inspect the files or state the uncertainty.", "Assume the file contents from memory."),
    pref("honesty", "If two instructions conflict, what should an assistant do?", "Follow the higher-priority or newer instruction and note the conflict if needed.", "Ignore the conflict and follow both."),
    pref("honesty", "If a command fails, what should the summary say?", "Report that it failed and include the relevant error.", "Say the command succeeded to keep the summary short."),
    pref("honesty", "If you used an assumption, what should you do?", "State the assumption clearly.", "Hide the assumption so the answer sounds cleaner."),
    pref("honesty", "If a model output looks wrong, what should you do?", "Treat it as evidence to investigate, not proof.", "Treat it as proof and move on."),
    pref("honesty", "If the user asks for a source you do not have, what should you do?", "Say you do not have a source.", "Invent a credible citation."),
    pref("honesty", "If you did not run a command, should you imply that you did?", "No.", "Yes, if the result is likely."),
    pref("honesty", "If a question asks for current information, what should you do?", "Check current information or state uncertainty.", "Answer from memory with confidence."),
    pref("honesty", "If a file path is missing, what should the assistant say?", "Ask for the path or explain that it is missing.", "Pretend it found the file."),
    # grounded context use, including course facts supplied in the prompt
    pref("grounding", "Context: Ada wrote the report on Monday. Who wrote the report?", "Ada wrote the report.", "Ben wrote the report."),
    pref("grounding", "Context: The meeting starts at 3 PM. When does the meeting start?", "It starts at 3 PM.", "It starts at noon."),
    pref("grounding", "Context: The blue key opens the lab. Which key opens the lab?", "The blue key opens the lab.", "The red key opens the lab."),
    pref("grounding", "Context: Module 14 covers DPO. Which module covers DPO?", "Module 14 covers DPO.", "Module 13 covers DPO."),
    pref("grounding", "Context: The file is named notes.md. What is the filename?", "notes.md", "summary.txt"),
    pref("grounding", "Context: The starter set has at least 150 preference pairs. What is the lower bound?", "150 preference pairs.", "120 preference pairs."),
    pref("grounding", "Context: The model ran on MPS. What device did it use?", "It used MPS.", "It used CUDA."),
    pref("grounding", "Context: The validation loss rose after step 800. What rose after step 800?", "The validation loss rose.", "The training batch size rose."),
    pref("grounding", "Context: The answer must be JSON only. What format is required?", "JSON only.", "Plain English paragraphs."),
    pref("grounding", "Context: The checkpoint path is artifacts/models/BaseLM-SFT. What is the checkpoint path?", "artifacts/models/BaseLM-SFT", "data/cache/BaseLM"),
    pref("grounding", "Context: Gradients tell parameters how to change to lower loss. What do gradients tell parameters?", "How to change to lower loss.", "How to store the training data."),
    pref("grounding", "Context: BPE creates reusable tokens for common text patterns. What does BPE create?", "Reusable tokens for common text patterns.", "Random bytes for rare words."),
    pref("grounding", "Context: Softmax turns logits into probabilities that sum to one. What does softmax produce?", "Probabilities that sum to one.", "Sorted logits."),
    pref("grounding", "Context: SFT masks user tokens and trains on assistant tokens. Which tokens does SFT train on?", "Assistant tokens.", "User tokens."),
    pref("grounding", "Context: DPO compares chosen and rejected responses relative to a frozen reference. What does DPO compare?", "Chosen and rejected responses.", "Two tokenizers."),
    pref("grounding", "Context: Retrieval adds external context before generation. What does retrieval add?", "External context.", "New model weights."),
    pref("grounding", "Context: A KV cache reuses past keys and values during inference. What does a KV cache reuse?", "Past keys and values.", "Optimizer gradients."),
    pref("grounding", "Context: The tokenizer artifact preserves the text-to-token mapping. What does the tokenizer artifact preserve?", "The text-to-token mapping.", "The trained model weights."),
    pref("grounding", "Context: AdamW decouples weight decay from the adaptive update. What does AdamW decouple?", "Weight decay.", "The validation set."),
    pref("grounding", "Context: A causal mask prevents positions from seeing future tokens. What does a causal mask prevent?", "Seeing future tokens.", "Seeing earlier tokens."),
    pref("grounding", "Context: Validation examples are not used for parameter updates. What are validation examples not used for?", "Parameter updates.", "Measuring generalization."),
    pref("grounding", "Context: DPO keeps a frozen reference model. What kind of reference model does DPO keep?", "A frozen reference model.", "A constantly updated reference model."),
    pref("grounding", "Context: Tool calls are structured text parsed by the runtime. What parses tool calls?", "The runtime.", "The optimizer."),
    pref("grounding", "Context: RAG retrieves relevant chunks before generation. What does RAG retrieve?", "Relevant chunks.", "New gradients."),
    # debugging advice
    pref("debugging", "Give one tip for debugging a failing test.", "Run the smallest failing test and inspect the first wrong value.", "Tests are annoying, so just change the code until it works."),
    pref("debugging", "What should I do if my training loss is NaN?", "Lower the learning rate and check for unstable operations or invalid data.", "Keep training; NaN usually fixes itself."),
    pref("debugging", "How do I check whether a model is overfitting?", "Compare train loss against validation loss over time.", "Only look at the final training loss."),
    pref("debugging", "What should I check after a tensor shape error?", "Print the relevant shapes and match them to the expected contract.", "Add random reshapes until the error disappears."),
    pref("debugging", "What should I do if MPS is unavailable?", "Confirm PyTorch sees MPS and fall back to CPU if needed.", "Assume the GPU is working because the Mac has Apple Silicon."),
    pref("debugging", "What should I do if a checkpoint is corrupted?", "Use the latest intact checkpoint or rerun from a clean save point.", "Keep loading the same corrupted file until it works."),
    pref("debugging", "What should I do if token IDs exceed the model vocabulary?", "Use the tokenizer that matches the model artifact.", "Clamp large token IDs to the final vocabulary entry."),
    pref("debugging", "What should I do if the notebook uses too much memory?", "Close other kernels and reduce batch size or context length.", "Open another notebook so the run has more space."),
    pref("debugging", "What should I log during a long training run?", "Track train loss, validation loss, learning rate, and sample outputs.", "Only log the final answer."),
    pref("debugging", "What should I do before changing many hyperparameters?", "Change one thing at a time and record the result.", "Change everything at once."),
    # concise answer style
    pref("concise_style", "Explain a tensor in one sentence.", "A tensor is an array of numbers with a shape.", "A tensor is a thing that is very important and can be many things in many ways."),
    pref("concise_style", "Explain gradient descent in one sentence.", "Gradient descent updates parameters in the direction that lowers loss.", "Gradient descent is when the model goes around and learns stuff until it gets better somehow."),
    pref("concise_style", "Explain a tokenizer in one sentence.", "A tokenizer converts text into token IDs a model can process.", "A tokenizer is a big complicated text thing that does all sorts of text processing."),
    pref("concise_style", "Explain logits in one sentence.", "Logits are unnormalized scores before softmax turns them into probabilities.", "Logits are numbers and then there is softmax and then it is kind of probability-like."),
    pref("concise_style", "Explain attention in one sentence.", "Attention lets each token mix information from relevant earlier tokens.", "Attention is when the model pays attention and does transformer things."),
    pref("concise_style", "Explain an embedding in one sentence.", "An embedding is a learned vector representation of a token.", "An embedding is a number thing that stores meaning in a mysterious way."),
    pref("concise_style", "Explain RAG in one sentence.", "RAG retrieves relevant context and includes it in the prompt before generation.", "RAG is when the model gets documents and is smarter."),
    pref("concise_style", "Explain a learning rate in one sentence.", "A learning rate controls how large each parameter update is.", "A learning rate is a setting you change when training feels wrong."),
    pref("concise_style", "Explain a validation set in one sentence.", "A validation set estimates performance on data not used for training updates.", "A validation set is extra training data you look at sometimes."),
    pref("concise_style", "Explain a tool call in one sentence.", "A tool call is structured text the runtime parses and executes.", "A tool call is when the model magically uses an external program."),
    pref("concise_style", "Explain DPO in one sentence.", "DPO trains a policy to prefer chosen responses over rejected ones while staying near a reference.", "DPO is a preference thing that makes answers better in some way."),
    pref("concise_style", "Explain SFT in one sentence.", "SFT trains a model on example prompts and desired assistant responses.", "SFT is when the model becomes an assistant somehow."),
    pref("concise_style", "Explain a system prompt in one sentence.", "A system prompt gives hidden instructions that frame assistant behavior.", "A system prompt is just the user's message with another name."),
    pref("concise_style", "Explain overfitting in one sentence.", "Overfitting means training improves while generalization gets worse.", "Overfitting is when the model trains for a long time."),
    # tool-call JSON shape
    pref("tool_json", "Call calculator for 23 * 17.", "{\"name\":\"calculator\",\"arguments\":{\"expression\":\"23*17\"}}", "calculator 23 times 17 please"),
    pref("tool_json", "Call calculator for 10 + 5.", "{\"name\":\"calculator\",\"arguments\":{\"expression\":\"10+5\"}}", "{\"tool\":\"calculator\",\"expr\":\"10+5\"}"),
    pref("tool_json", "Call search for gradient clipping.", "{\"name\":\"search\",\"arguments\":{\"query\":\"gradient clipping\"}}", "Search for gradient clipping."),
    pref("tool_json", "Call read_file for docs/syllabus.md.", "{\"name\":\"read_file\",\"arguments\":{\"path\":\"docs/syllabus.md\"}}", "{\"name\":\"read_file\",\"path\":\"docs/syllabus.md\"}"),
    pref("tool_json", "Call list_files for docs/modules.", "{\"name\":\"list_files\",\"arguments\":{\"path\":\"docs/modules\"}}", "{name:list_files,args:docs/modules}"),
    pref("tool_json", "Call calculator for 7 squared.", "{\"name\":\"calculator\",\"arguments\":{\"expression\":\"7*7\"}}", "The answer is probably 49."),
    pref("tool_json", "Call lookup_weather for Boston.", "{\"name\":\"lookup_weather\",\"arguments\":{\"location\":\"Boston\"}}", "{\"name\":\"lookup_weather\",\"location\":\"Boston\"}"),
    pref("tool_json", "Call python_eval for len('abc').", "{\"name\":\"python_eval\",\"arguments\":{\"code\":\"len('abc')\"}}", "{\"name\":\"python_eval\",\"arguments\":\"len('abc')\"}"),
    pref("tool_json", "Call summarize_file for README.md.", "{\"name\":\"summarize_file\",\"arguments\":{\"path\":\"README.md\"}}", "I will summarize README.md now."),
    pref("tool_json", "Call grep for TODO in g2c.", "{\"name\":\"grep\",\"arguments\":{\"pattern\":\"TODO\",\"path\":\"g2c\"}}", "{\"grep\":\"TODO\",\"where\":\"g2c\"}"),
    pref("tool_json", "Call calculator for 18 / 3.", "{\"name\":\"calculator\",\"arguments\":{\"expression\":\"18/3\"}}", "{\"name\":\"calculator\",\"arguments\":\"18/3\"}"),
    pref("tool_json", "Call search for module 14 DPO.", "{\"name\":\"search\",\"arguments\":{\"query\":\"module 14 DPO\"}}", "module 14 DPO search"),
    pref("tool_json", "Call calculator for 12 * 12.", "{\"name\":\"calculator\",\"arguments\":{\"expression\":\"12*12\"}}", "{\"name\":\"calculator\",\"expression\":\"12*12\"}"),
    pref("tool_json", "Call search for tokenization.", "{\"name\":\"search\",\"arguments\":{\"query\":\"tokenization\"}}", "search tokenization"),
    pref("tool_json", "Call read_file for README.md.", "{\"name\":\"read_file\",\"arguments\":{\"path\":\"README.md\"}}", "Read README.md."),
    pref("tool_json", "Call list_files for data.", "{\"name\":\"list_files\",\"arguments\":{\"path\":\"data\"}}", "{\"name\":\"list_files\",\"path\":\"data\"}"),
    pref("tool_json", "Call calculator for 5 + 8.", "{\"name\":\"calculator\",\"arguments\":{\"expression\":\"5+8\"}}", "The calculator result is 13."),
    pref("tool_json", "Call grep for FIXME in tests.", "{\"name\":\"grep\",\"arguments\":{\"pattern\":\"FIXME\",\"path\":\"tests\"}}", "{\"grep\":{\"pattern\":\"FIXME\",\"path\":\"tests\"}}"),
    # answering the actual question
    pref("actual_question", "What command runs the tests?", "pytest", "Testing is useful because it catches bugs."),
    pref("actual_question", "Which section comes after The big idea?", "Concepts to internalize.", "The big idea explains the module."),
    pref("actual_question", "Answer the final word of this sentence: models learn from data.", "data.", "Models can learn many patterns."),
    pref("actual_question", "Give the file extension of notebook.ipynb.", ".ipynb", "A notebook is an interactive document."),
    pref("actual_question", "What does SFT stand for?", "Supervised fine-tuning.", "SFT is used after pretraining."),
    pref("actual_question", "What is the requested output format: JSON or markdown?", "JSON.", "Both JSON and markdown are common formats."),
    pref("actual_question", "How many items are in this list: A, B, C?", "Three.", "The list contains letters."),
    pref("actual_question", "What is the first token marker in '<|user|>'?", "<|user|>", "It is a chat marker."),
    pref("actual_question", "What is the last word in 'the model predicts tokens'?", "tokens", "The sentence is about models."),
    pref("actual_question", "Which is larger: 9 or 4?", "9.", "Both are numbers."),
    pref("actual_question", "What file extension do Python files use?", ".py", "Python files are used for code."),
    pref("actual_question", "What word comes after 'machine' in 'machine learning'?", "learning", "Machine learning is important."),
    pref("actual_question", "How many letters are in cat?", "Three.", "Cat is an animal."),
    pref("actual_question", "Which word is repeated: blue red blue?", "blue", "The colors are blue and red."),
    pref("actual_question", "What is the middle item in red, green, blue?", "green", "The list contains colors."),
    pref("actual_question", "What is the final token marker in '<|end|>'?", "<|end|>", "It marks the end of a message."),
    pref("actual_question", "Which is smaller: 3 or 8?", "3.", "Both are integers."),
    pref("actual_question", "What is the first word in 'small language model'?", "small", "The phrase describes a model."),
]

print(f"preference rows: {len(preference_rows)}")
print(json.dumps(preference_rows[0], indent=2))
assert len(preference_rows) >= 140

Save your preference rows if you want the dataset as a reusable artifact. The notebook keeps this disabled by default so exploratory edits do not overwrite a hand-curated file.

In [ ]:
SAVE_PREFERENCE_JSON = False
PREFERENCE_JSON_PATH = repo_root / "data" / "dpo" / "preferences.json"

if SAVE_PREFERENCE_JSON:
    PREFERENCE_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
    PREFERENCE_JSON_PATH.write_text(
        json.dumps(preference_rows, indent=2) + "\n",
        encoding="utf-8",
    )
    print("saved", PREFERENCE_JSON_PATH.relative_to(repo_root))
else:
    print("not saved; set SAVE_PREFERENCE_JSON = True when the dataset is ready")

Check token lengths before training. If chosen completions are systematically longer, DPO can learn length instead of preference.

In [ ]:
def response_ids(text: str) -> list[int]:
    return tokenizer.encode_with_vocab_size(text + template.END, policy_model.vocab_size)

length_rows = []
for row in preference_rows:
    chosen_len = len(response_ids(row["chosen"]))
    rejected_len = len(response_ids(row["rejected"]))
    ratio = max(chosen_len, rejected_len) / max(1, min(chosen_len, rejected_len))
    length_rows.append((row["kind"], chosen_len, rejected_len, ratio, row["user"]))

print(f"{'kind':<12} {'chosen':>6} {'rejected':>8} {'ratio':>6}  prompt")
for kind, chosen_len, rejected_len, ratio, user in length_rows[:20]:
    print(f"{kind:<12} {chosen_len:>6} {rejected_len:>8} {ratio:>6.2f}  {user[:54]}")

avg_chosen = sum(row[1] for row in length_rows) / len(length_rows)
avg_rejected = sum(row[2] for row in length_rows) / len(length_rows)
print("\naverage chosen tokens:", round(avg_chosen, 2))
print("average rejected tokens:", round(avg_rejected, 2))
print("max length ratio:", round(max(row[3] for row in length_rows), 2))

## Encode preference triples

A DPO prompt is the user turn plus the assistant role marker. The chosen and rejected responses are completions after that marker, and both include `<|end|>` so the model can learn which answer should stop.

In [ ]:
def render_dpo_prompt(user_text: str) -> str:
    return template.render([{"role": "user", "content": user_text}]) + f"{template.ASSISTANT}\n"


def encode_preference(row: dict) -> PreferenceExample:
    prompt_ids = tokenizer.encode_with_vocab_size(
        render_dpo_prompt(row["user"]),
        policy_model.vocab_size,
    )
    chosen_ids = tokenizer.encode_with_vocab_size(
        row["chosen"] + template.END,
        policy_model.vocab_size,
    )
    rejected_ids = tokenizer.encode_with_vocab_size(
        row["rejected"] + template.END,
        policy_model.vocab_size,
    )
    return PreferenceExample(
        prompt_ids=prompt_ids,
        chosen_ids=chosen_ids,
        rejected_ids=rejected_ids,
    )

encoded_rows = [(row, encode_preference(row)) for row in preference_rows]
encoded_preferences = [ex for _, ex in encoded_rows]

first_row, first_ex = encoded_rows[0]
print("prompt text:")
print(render_dpo_prompt(first_row["user"]))
print("prompt ids:", len(first_ex.prompt_ids))
print("chosen ids:", len(first_ex.chosen_ids), tokenizer.decode(first_ex.chosen_ids))
print("rejected ids:", len(first_ex.rejected_ids), tokenizer.decode(first_ex.rejected_ids))
assert max(max(ex.prompt_ids + ex.chosen_ids + ex.rejected_ids) for ex in encoded_preferences) < policy_model.vocab_size

Inspect the collator on two examples. The mask should be `1` only on response targets, never on the prompt or padding.

In [ ]:
DPO_MAX_SEQ_LEN = min(128, policy_model.max_seq_len)

cx, cy, cm, rx, ry, rm = pad_and_collate_pref(
    encoded_preferences[:2],
    max_seq_len=DPO_MAX_SEQ_LEN,
    pad_id=pad_id,
)
print("chosen x/y/mask:", cx.shape, cy.shape, cm.shape)
print("rejected x/y/mask:", rx.shape, ry.shape, rm.shape)
print("chosen mask sums:", cm.sum(dim=1).tolist())
print("rejected mask sums:", rm.sum(dim=1).tolist())


def masked_target_text(y: torch.Tensor, mask: torch.Tensor) -> str:
    ids = [int(token_id) for token_id, keep in zip(y.tolist(), mask.tolist()) if int(keep) == 1]
    return tokenizer.decode(ids)

print("\nchosen target text:", masked_target_text(cy[0], cm[0]))
print("rejected target text:", masked_target_text(ry[0], rm[0]))

## Train/validation split

In [ ]:
generator = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(encoded_rows), generator=generator).tolist()
VAL_FRACTION = 0.25
val_count = min(len(perm) - 1, max(1, round(VAL_FRACTION * len(perm))))
train_indices = perm[:-val_count]
val_indices = perm[-val_count:]

train_rows = [encoded_rows[i][0] for i in train_indices]
val_rows = [encoded_rows[i][0] for i in val_indices]
train_examples = [encoded_rows[i][1] for i in train_indices]
val_examples = [encoded_rows[i][1] for i in val_indices]

print("train examples:", len(train_examples))
print("val examples:", len(val_examples))

def average(values):
    return sum(values) / len(values) if values else 0.0


def percent(numerator: int, denominator: int) -> float:
    return 100.0 * numerator / denominator if denominator else 0.0


chosen_lengths = [len(ex.chosen_ids) for ex in encoded_preferences]
rejected_lengths = [len(ex.rejected_ids) for ex in encoded_preferences]
chosen_longer = sum(c > r for c, r in zip(chosen_lengths, rejected_lengths))
train_kinds = [encoded_rows[i][0]["kind"] for i in train_indices]
val_kinds = [encoded_rows[i][0]["kind"] for i in val_indices]
all_kinds = sorted({row["kind"] for row in preference_rows})

print("\npreference dataset summary")
print(f"{'metric':<30} value")
print("-" * 44)
print(f"{'pairs':<30} {len(encoded_preferences):>6}")
print(f"{'train pairs':<30} {len(train_examples):>6}")
print(f"{'validation pairs':<30} {len(val_examples):>6}")
print(f"{'avg chosen response tokens':<30} {average(chosen_lengths):>6.1f}")
print(f"{'avg rejected response tokens':<30} {average(rejected_lengths):>6.1f}")
print(f"{'chosen longer':<30} {percent(chosen_longer, len(encoded_preferences)):>5.1f}%")

print("\nkind distribution")
print(f"{'kind':<18} {'total':>5} {'train':>5} {'val':>5}")
print("-" * 36)
for kind in all_kinds:
    total = sum(row["kind"] == kind for row in preference_rows)
    train = sum(item == kind for item in train_kinds)
    val = sum(item == kind for item in val_kinds)
    print(f"{kind:<18} {total:>5} {train:>5} {val:>5}")

## Exercise 2 - Step-0 DPO sanity check

Before training, the policy and reference are identical copies. The DPO margin is therefore zero and the loss should be `log(2) ≈ 0.6931`. This is the deepest single sanity check for your data path.

In [ ]:
def preference_batch_metrics(policy, reference, examples, *, beta: float, max_seq_len: int):
    cx, cy, cm, rx, ry, rm = pad_and_collate_pref(
        examples,
        max_seq_len=max_seq_len,
        pad_id=pad_id,
    )
    device = model_device(policy)
    cx, cy, cm = cx.to(device), cy.to(device), cm.to(device)
    rx, ry, rm = rx.to(device), ry.to(device), rm.to(device)
    with torch.no_grad():
        policy_c = sequence_logprob(policy(cx), cy, cm)
        policy_r = sequence_logprob(policy(rx), ry, rm)
        ref_c = sequence_logprob(reference(cx), cy, cm)
        ref_r = sequence_logprob(reference(rx), ry, rm)
        loss, metrics = dpo_loss(policy_c, policy_r, ref_c, ref_r, beta=beta)
    return {
        "loss": loss.item(),
        "chosen_reward": metrics["chosen_reward"].item(),
        "rejected_reward": metrics["rejected_reward"].item(),
        "reward_margin": metrics["reward_margin"].item(),
        "accuracy": metrics["accuracy"].item(),
    }

initial_metrics = preference_batch_metrics(
    policy_model,
    ref_model,
    train_examples[: min(8, len(train_examples))],
    beta=0.1,
    max_seq_len=DPO_MAX_SEQ_LEN,
)
print(initial_metrics)
print("log(2):", math.log(2))
assert abs(initial_metrics["loss"] - math.log(2)) < 1e-3

## Exercise 3 - Train DPO

The defaults are still conservative, but the larger preference set benefits from a few more steps. DPO is about 3x the wall-clock of SFT at the same model size because it runs policy/reference on chosen/rejected sequences. Increase steps only after the loss, reward margin, and samples make sense.

In [ ]:
IS_HF_ARTIFACT = policy_artifact.manifest.get("kind") == "huggingface_causal_lm"

DPO_CONFIG = {
    "max_seq_len": DPO_MAX_SEQ_LEN,
    "pad_id": pad_id,
    "beta": 0.1,
    "batch_size": 1 if IS_HF_ARTIFACT else 4,
    "max_steps": 240 if IS_HF_ARTIFACT else 500,
    "max_lr": 5e-5 if IS_HF_ARTIFACT else 1e-4,
    "min_lr": 5e-6 if IS_HF_ARTIFACT else 1e-5,
    "warmup_steps": 10,
    "weight_decay": 0.0,
    "grad_clip": 1.0,
    "eval_every": 20 if IS_HF_ARTIFACT else 50,
    "eval_iters": 3,
    "log_every": 5 if IS_HF_ARTIFACT else 10,
    "device": TRAIN_DEVICE,
}
DPO_CONFIG

In [ ]:
trainer = DPOTrainer(
    policy_model,
    ref_model=ref_model,
    examples=train_examples,
    generator=torch.Generator().manual_seed(SEED),
    **DPO_CONFIG,
)

history = train_dpo_with_progress(
    f"{policy_artifact.name} DPO",
    trainer,
    eval_examples=val_examples,
)
plot_dpo_history(history)

## Exercise 4 - Compare SFT vs DPO behavior

Use the same seed and sampling settings. The reference model is your original SFT checkpoint; the policy model is now DPO-tuned.

The first group is **targeted failure probes**: prompts whose rejected answer resembles a bad SFT sample. These are allowed to overlap with preference training because they show whether DPO corrected the specific behavior it was asked to correct.

The second group is **heldout probes**: similar tasks, but not exact preference rows. These test whether the preference shift generalizes beyond the examples.

Free samples are interesting, but they are not the most direct DPO diagnostic. DPO trains pairwise preferences: it can make the chosen completion score higher than the rejected completion without changing the top sampled answer for an open-ended prompt. The score table after the samples is often the clearer view of what changed.

In [ ]:
TARGETED_FAILURE_PROMPTS = [
    "Which planet is known as the red planet?",
    "What is 7 + 5?",
    "Explain logits in one sentence.",
    "Explain RAG in one sentence.",
    "Explain a tool call in one sentence.",
    "Context: Softmax turns logits into probabilities that sum to one. What does softmax produce?",
    "If you are unsure about an answer, what should you say?",
]

HELDOUT_PROMPTS = [
    # factual/arithmetic heldouts
    "What is the capital of Russia?",
    "What is the capital of Portugal?",
    "Which planet is closest to the Sun?",
    "What continent is Brazil in?",
    "What is the capital of Canada?",
    "What is the capital of Australia?",
    "What is the chemical symbol for water?",
    "How many continents are there?",
    "Who wrote Romeo and Juliet?",
    "What is the boiling point of water in Celsius?",
    "Which star is closest to Earth?",
    "What is the main gas in Earth's atmosphere?",
    "What is the largest mammal?",
    "What gas do humans breathe in to survive?",
    "What is 9 + 4?",
    "What is 8 * 9?",
    # format/actual question heldouts
    "Answer in one word: opposite of loud.",
    "Return exactly one animal.",
    "Return JSON with city Paris.",
    "Return exactly two bullet items: alpha and beta.",
    "What is the last word in 'tokens become vectors'?",
    "What file extension do markdown files use?",
    # grounded context heldouts
    "Context: The run used CPU. What device did it use?",
    "Context: A causal mask blocks future tokens. What does the mask block?",
    "Context: The answer must be markdown. What format is required?",
    "Context: Nora fixed the bug. Who fixed the bug?",
    # debugging/honesty heldouts
    "If local files may contain the answer, what should an assistant do before answering?",
    "What should I log during a long training run?",
    "What should I check after an out-of-vocabulary token error?",
    "Should an assistant invent missing evidence? Answer yes or no.",
    # tool heldouts
    "Call calculator for 8 * 9.",
    "Call search for validation loss.",
    "Call read_file for README.md.",
    "Call list_files for data.",
]


def compare_models(title: str, prompts: list[str], *, seed: int = SEED) -> None:
    print("\n" + title)
    for i, prompt in enumerate(prompts, start=1):
        print("=" * 72)
        print(f"[{i}/{len(prompts)}] user:", prompt)
        show_response("SFT/reference", ref_model, prompt, max_new_tokens=80, seed=seed)
        show_response("DPO/policy", policy_model, prompt, max_new_tokens=80, seed=seed)

compare_models("Targeted SFT failure probes", TARGETED_FAILURE_PROMPTS)
compare_models("Heldout preference probes", HELDOUT_PROMPTS)

Score held-out preference pairs directly. Positive reward margin means the DPO policy favors the chosen completion over the rejected completion more than the frozen SFT reference does. This is the most direct view of the DPO objective.

In [ ]:
def score_preference_example(policy, reference, ex: PreferenceExample, *, beta: float = 0.1) -> dict[str, float]:
    metrics = preference_batch_metrics(policy, reference, [ex], beta=beta, max_seq_len=DPO_MAX_SEQ_LEN)
    return metrics

print(f"{'kind':<16} {'margin':>8} {'acc':>5}  {'chosen':<32} {'rejected':<32} prompt")
for row, ex in zip(val_rows, val_examples):
    metrics = score_preference_example(policy_model, ref_model, ex, beta=DPO_CONFIG["beta"])
    chosen = row['chosen'].replace("\n", " ")[:32]
    rejected = row['rejected'].replace("\n", " ")[:32]
    print(
        f"{row['kind']:<16} {metrics['reward_margin']:>8.3f} {metrics['accuracy']:>5.2f}  "
        f"{chosen:<32} {rejected:<32} {row['user'][:52]}"
    )

### Written reflection

Question: Where did DPO clearly improve the SFT model, and where did it fail or make behavior worse?

Answer: 

## Exercise 5 - Optional beta sweep

Run this only after the baseline run works. The quickest useful sweep is three betas over fewer steps. The goal is not a perfect model; it is to see low beta drift, mid beta learning, and high beta under-movement.

Trains three fresh policies, so expect a few minutes. Skip the cell if you do not want to wait.

In [ ]:
BETA_VALUES = [0.05, 0.1, 0.3]
BETA_SWEEP_STEPS = 80

beta_results = {}
for beta in BETA_VALUES:
    fresh_policy = load_model_artifact_with_tokenizer(
        SFT_ARTIFACT_NAME,
        repo_root=repo_root,
        device=TRAIN_DEVICE,
    ).model
    fresh_ref = load_model_artifact_with_tokenizer(
        SFT_ARTIFACT_NAME,
        repo_root=repo_root,
        device=TRAIN_DEVICE,
    ).model
    config = {**DPO_CONFIG, "beta": beta, "max_steps": BETA_SWEEP_STEPS}
    sweep_trainer = DPOTrainer(
        fresh_policy,
        ref_model=fresh_ref,
        examples=train_examples,
        generator=torch.Generator().manual_seed(SEED),
        **config,
    )
    sweep_history = train_dpo_with_progress(
        f"beta={beta}",
        sweep_trainer,
        eval_examples=val_examples,
    )
    beta_results[beta] = {
        "history": sweep_history,
        "model": fresh_policy,
    }

for beta, result in beta_results.items():
    h = result["history"]
    print(
        beta,
        "final loss", round(h["train_loss"][-1], 4),
        "final margin", round(h["reward_margin"][-1], 4),
        "final acc", round(h["accuracy"][-1], 3),
    )

### Beta sweep notes

Question: Which beta moved the model enough without visibly damaging its base behavior?

Answer: 

## Exercise 6 - Save the DPO artifact

This preserves the original SFT artifact and writes a separate DPO artifact for Module 15 evaluation. If your samples got worse, set `SAVE_DPO_ARTIFACT = False`, tune the run, and save only when the artifact is worth reusing.

In [ ]:
DPO_ARTIFACT_NAME = (
    SFT_ARTIFACT_NAME[:-4] + "-DPO"
    if SFT_ARTIFACT_NAME.endswith("-SFT")
    else f"{SFT_ARTIFACT_NAME}-DPO"
)
SAVE_DPO_ARTIFACT = True

if SAVE_DPO_ARTIFACT:
    training_config = {
        **DPO_CONFIG,
        "sft_artifact": SFT_ARTIFACT_NAME,
        "num_examples": len(encoded_preferences),
        "num_train_examples": len(train_examples),
        "num_val_examples": len(val_examples),
        "preference_rows": len(preference_rows),
    }
    if policy_artifact.manifest.get("kind") == "huggingface_causal_lm":
        artifact_dir = save_huggingface_model_artifact(
            DPO_ARTIFACT_NAME,
            model=policy_model,
            tokenizer=tokenizer,
            base_artifact_name=SFT_ARTIFACT_NAME,
            training_config=training_config,
            source=f"DPO on {len(encoded_preferences)} preference pairs from {SFT_ARTIFACT_NAME}",
            history=history,
            module="module-14",
            notes="Preference-tuned checkpoint for Module 15 evaluation experiments.",
            repo_root=repo_root,
        )
    else:
        model_config = dict(policy_artifact.manifest["model_config"])
        artifact_dir = save_model_artifact(
            DPO_ARTIFACT_NAME,
            model=policy_model,
            model_config=model_config,
            training_config=training_config,
            tokenizer_artifact_name=policy_artifact.manifest["tokenizer_artifact"],
            source=f"DPO on {len(encoded_preferences)} preference pairs from {SFT_ARTIFACT_NAME}",
            history=history,
            seed=SEED,
            module="module-14",
            notes="Preference-tuned checkpoint for Module 15 evaluation experiments.",
            repo_root=repo_root,
        )
    print("saved", artifact_dir.relative_to(repo_root))
else:
    print("not saved")

## Deliverable notes

Question: Explain why the initial DPO loss is `log(2)`.

Answer: 

Question: Explain why the reference model must stay frozen.

Answer: 

Question: What is one preference-dataset bias you checked for?

Answer: 